# 04 — Dieci ripetizioni: media e variabilità delle prestazioni

Confrontiamo **max_depth=10, 13 e 15**, ciascuno su **10 split stratificati 80/20**
ricavati da KDDTrain+. Tutte le 41 feature restano conservate. In ogni ripetizione
i tre alberi usano esattamente lo stesso train e la stessa validation.

Cambiano fra ripetizioni lo split e la casualità dell'albero. Riportiamo media,
deviazione standard campionaria, minimo e massimo: la media da sola nasconderebbe
l'instabilità sulle classi rare. Questo è un controllo di sensibilità alla casualità,
non dieci nuovi dataset e non una valutazione del test ufficiale.


In [ ]:
from pathlib import Path
import sys
import hashlib
import json
import platform
import importlib.metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

project_root = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(project_root))
from src.data import COLUMN_NAMES, FEATURE_COLUMNS, CATEGORICAL_COLUMNS
from src.targets import ATTACK_TO_FAMILY, FAMILY_ORDER


## 1. Leggere il training e definire X/y

Usiamo direttamente pandas. Dal codice locale importiamo soltanto le costanti
dello schema e la mappa delle famiglie. Nessuna riga o feature viene eliminata;
target e difficulty non entrano in X. KDDTest+ non viene caricato.


In [ ]:
train_path = project_root / 'data/raw/KDDTrain+.csv'
string_types = {column: 'string' for column in (*CATEGORICAL_COLUMNS, 'attack_label')}
data = pd.read_csv(train_path, header=None, names=COLUMN_NAMES, dtype=string_types)
X = data.loc[:, list(FEATURE_COLUMNS)].copy()
unknown = data['attack_label'].isna() | ~data['attack_label'].isin(ATTACK_TO_FAMILY)
if unknown.any():
    raise ValueError('Etichette mancanti o non mappate: riesaminare il target.')
y = data['attack_label'].map(ATTACK_TO_FAMILY).rename('family')
assert X.index.equals(pd.RangeIndex(len(X)))
assert X.shape[1] == 41 and not X.isna().any().any()
assert np.isfinite(X.select_dtypes(include='number').to_numpy()).all()
print(f'Training ufficiale: {len(X):,} righe; {X.shape[1]} feature.')


## 2. Dieci seed diversi, registrati e riproducibili

Un `random_state` intero sempre uguale riproduce la stessa casualità. Qui lo
cambiamo a ogni ripetizione: **42, 43, ..., 51**, fissati prima di vedere i risultati.
Lo passiamo sia a `train_test_split` sia al classificatore. Non serve usare `None`
per ottenere prove diverse: conservare i seed ci permette di ricreare ciascuna prova.
Variare entrambe le sorgenti misura la loro variabilità combinata, senza separarle.
[Riferimento sklearn sulla casualità](https://scikit-learn.org/stable/common_pitfalls.html#controlling-randomness).

Il primo seed, 42, coincide con il confronto precedente e consente una verifica.
Preprocessing e modello sono definiti ora, ma vengono appresi soltanto nel train
di ciascuna ripetizione, tramite una nuova Pipeline clonata con sklearn.


In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

seeds = list(range(1, 6))
depths = [10, 13, 15]
validation_size = 0.2
numeric_columns = [column for column in FEATURE_COLUMNS if column not in CATEGORICAL_COLUMNS]
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), list(CATEGORICAL_COLUMNS)),
    ('numeric', 'passthrough', numeric_columns),
], remainder='drop')
model_template = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier()),
])
print('Seed:', seeds)
print(f'{len(seeds)} ripetizioni × {len(depths)} profondità = {len(seeds) * len(depths)} fit.')


## 3. Ripetere split → fit → predict → metriche

Il ciclo esterno crea uno split per seed; quello interno confronta le tre profondità
sulle stesse righe. Ogni Pipeline è nuova: nessun encoder o albero passa informazioni
da un train al successivo. Salviamo anche indici, risposte e predizioni, per controllare
le misure senza riaddestrare. Non selezioniamo o scartiamo ripetizioni.


In [ ]:
run_rows = []
class_rows = []
normalized_matrices = {depth: [] for depth in depths}
count_matrices = {depth: [] for depth in depths}
prediction_arrays = {}
split_records = []

for run, seed in enumerate(seeds, start=1):
    # Uno split condiviso dai tre alberi della stessa ripetizione.
    X_train, X_validation, y_train, y_validation = train_test_split(
        X, y, test_size=validation_size, stratify=y, random_state=seed,
    )
    train_indices = X_train.index.to_numpy()
    validation_indices = X_validation.index.to_numpy()
    assert not np.intersect1d(train_indices, validation_indices).size
    assert set(y_train.unique()) == set(y_validation.unique()) == set(FAMILY_ORDER)
    prediction_arrays[f'run_{run:02d}_validation_indices'] = validation_indices
    prediction_arrays[f'run_{run:02d}_truth'] = np.asarray(y_validation, dtype=str)
    split_records.append({
        'run': run, 'seed': seed, 'train_rows': len(X_train),
        'validation_rows': len(X_validation),
        'train_indices_sha256': hashlib.sha256(train_indices.tobytes()).hexdigest(),
        'validation_indices_sha256': hashlib.sha256(validation_indices.tobytes()).hexdigest(),
    })

    for depth in depths:
        model = clone(model_template).set_params(
            classifier__max_depth=depth, classifier__random_state=seed,
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_validation)
        prediction_arrays[f'run_{run:02d}_depth_{depth}'] = np.asarray(y_pred, dtype=str)
        report = classification_report(
            y_validation, y_pred, labels=FAMILY_ORDER, output_dict=True, zero_division=0,
        )
        counts = confusion_matrix(y_validation, y_pred, labels=FAMILY_ORDER)
        normalized = confusion_matrix(y_validation, y_pred, labels=FAMILY_ORDER, normalize='true')
        count_matrices[depth].append(counts)
        normalized_matrices[depth].append(normalized)
        run_rows.append({
            'run': run, 'seed': seed, 'max_depth': depth,
            'accuracy': accuracy_score(y_validation, y_pred),
            'macro_f1': f1_score(y_validation, y_pred, labels=FAMILY_ORDER, average='macro', zero_division=0),
            'weighted_f1': f1_score(y_validation, y_pred, labels=FAMILY_ORDER, average='weighted', zero_division=0),
        })
        for i, family in enumerate(FAMILY_ORDER):
            support = int(counts[i].sum())
            correct = int(counts[i, i])
            false_positives = int(counts[:, i].sum() - correct)
            class_rows.append({
                'run': run, 'seed': seed, 'max_depth': depth, 'famiglia': family,
                'support': support, 'corretti': correct, 'errori': support - correct,
                'falsi_positivi': false_positives,
                'quota_errori': (support - correct) / support,
                'precision': report[family]['precision'],
                'recall': report[family]['recall'], 'f1': report[family]['f1-score'],
            })
    print(f'Ripetizione {run}/5 completata (seed={seed}).')

run_results = pd.DataFrame(run_rows)
class_results = pd.DataFrame(class_rows)
assert len(run_results) == 15 and len(class_results) == 75
display(run_results.round(6))


## 4. Medie e dispersione: non soltanto la media

Per ogni profondità diamo lo stesso peso alle dieci ripetizioni. `std` è la
**deviazione standard campionaria** (`ddof=1`), non l'errore standard della media
né un intervallo di confidenza. Minimo e massimo mostrano l'ampiezza osservata.
Precision e F1 sono calcolati in ogni ripetizione e poi mediati: non li ricaviamo
dalla matrice media, perché questi aggregati non sono intercambiabili.


In [ ]:
overall_summary = run_results.groupby('max_depth')[['accuracy', 'macro_f1', 'weighted_f1']].agg(
    ['mean', 'std', 'min', 'max'],
)
class_summary = class_results.groupby(['max_depth', 'famiglia'], sort=False)[
    ['support', 'errori', 'falsi_positivi', 'quota_errori', 'precision', 'recall', 'f1']
].agg(['mean', 'std', 'min', 'max'])
display(overall_summary.round(6))
rare_summary = class_summary.loc[(slice(None), ['R2L', 'U2R']), :]
display(rare_summary.loc[:, ['quota_errori', 'precision', 'recall', 'f1']].round(4))


## 5. Confusion matrix normalizzate medie

Prima normalizziamo ogni matrice per famiglia vera, poi facciamo la media
delle dieci matrici. Ogni riga della matrice media somma a 100% prima
dell'arrotondamento. La diagonale è il recall medio; le altre celle mostrano
dove finiscono gli errori. Usiamo la stessa scala cromatica nei tre pannelli.
Una quota media non è il conteggio di una singola prova: i conteggi e i falsi
positivi di ciascuna ripetizione restano nel CSV e nel riepilogo per famiglia.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

mean_matrices = {}
std_matrices = {}
fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
for ax, depth in zip(axes, depths):
    matrices = np.stack(normalized_matrices[depth])
    mean_matrices[depth] = matrices.mean(axis=0)
    std_matrices[depth] = matrices.std(axis=0, ddof=1)
    ConfusionMatrixDisplay(mean_matrices[depth], display_labels=FAMILY_ORDER).plot(
        values_format='.1%', cmap='Blues', colorbar=False,
        im_kw={'vmin': 0, 'vmax': 1}, ax=ax,
    )
    ax.set(title=f'max_depth = {depth}', xlabel='Famiglia prevista', ylabel='Famiglia vera')
fig.suptitle('Media di 10 confusion matrix normalizzate — validation interna')
plt.show()


## 6. Quanti esempi diversi stiamo osservando?

Le validation si sovrappongono: lo stesso record può essere valutato in più
ripetizioni e può apparire nel train di un'altra prova. Non trasferiamo modelli
fra le prove, ma le dieci misure non sono indipendenti. Contiamo anche i record
U2R **distinti** visti in validation, senza considerarli dieci gruppi nuovi di casi.


In [ ]:
unique_validation_indices = np.unique(np.concatenate([
    prediction_arrays[f'run_{run:02d}_validation_indices'] for run in range(1, len(seeds) + 1)
]))
unique_validation_counts = y.iloc[unique_validation_indices].value_counts().reindex(FAMILY_ORDER)
print('Record distinti apparsi almeno una volta in validation:')
display(unique_validation_counts.rename('record distinti'))
print('Support U2R per ripetizione:')
display(class_results.loc[
    (class_results['famiglia'] == 'U2R') & (class_results['max_depth'] == depths[0]),
    ['run', 'seed', 'support'],
])


## 7. Salvare risultati, matrici e predizioni

I risultati di ogni run sono conservati, inclusi quelli peggiori. Gli artefatti
sono in `artifacts/repeated_validation/`, ignorata da Git. I seed e l'hash del CSV
identificano il protocollo; i file del confronto sul singolo split non cambiano.


In [ ]:
artifacts = project_root / 'artifacts/repeated_validation'
artifacts.mkdir(parents=True, exist_ok=True)
run_results.to_csv(artifacts / 'run_metrics.csv', index=False)
class_results.to_csv(artifacts / 'class_metrics.csv', index=False)
overall_summary.to_csv(artifacts / 'overall_summary.csv')
class_summary.to_csv(artifacts / 'class_summary.csv')
np.savez_compressed(artifacts / 'predictions.npz', **prediction_arrays)
fig.savefig(artifacts / 'mean_confusion_matrices.png', dpi=160)

repeated_experiment = {
    'evaluation_set': 'repeated validation from KDDTrain+ only',
    'source': 'data/raw/KDDTrain+.csv',
    'source_sha256': hashlib.sha256(train_path.read_bytes()).hexdigest(),
    'rows': len(X), 'feature_count': X.shape[1], 'validation_size': validation_size,
    'seeds': seeds, 'depths': depths, 'stratified': True,
    'randomness_varied': ['train_validation_split', 'decision_tree'],
    'family_order': list(FAMILY_ORDER), 'std_ddof': 1,
    'versions': {'python': platform.python_version(), **{
        package: importlib.metadata.version(package)
        for package in ['numpy', 'pandas', 'scikit-learn', 'matplotlib']
    }},
    'splits': split_records,
    'run_metrics': run_results.to_dict(orient='records'),
    'class_metrics': class_results.to_dict(orient='records'),
    'normalized_confusion_mean': {str(depth): matrix.tolist() for depth, matrix in mean_matrices.items()},
    'normalized_confusion_std': {str(depth): matrix.tolist() for depth, matrix in std_matrices.items()},
    'confusion_counts_by_depth': {str(depth): [matrix.tolist() for matrix in matrices] for depth, matrices in count_matrices.items()},
    'unique_validation_counts': {family: int(value) for family, value in unique_validation_counts.items()},
    'limits': [
        'Overlapping repeated holdouts; measurements are not independent.',
        'Rare U2R cases are reused; repetitions do not create new observations.',
        'Known duplicate-feature overlap is not removed; see DATA_AUDIT.md.',
        'Exploration of three depths; no official-test performance estimate.',
    ],
}
(artifacts / 'results.json').write_text(json.dumps(repeated_experiment, indent=2), encoding='utf-8')
print('Salvati 15 fit, 75 righe per famiglia e tutte le predizioni, senza serializzare modelli.')


## Esito delle dieci prove

DS = deviazione standard campionaria fra le ripetizioni; pp = punti percentuali.

| Profondità | Errori R2L: media ± DS | Errori U2R: media ± DS | Macro F1: media ± DS |
|---|---:|---:|---:|
| 10 | 19.45% ± 2.77 pp | 55.00% ± 15.09 pp | 0.8851 ± 0.0282 |
| 13 | 8.74% ± 2.15 pp | 49.00% ± 18.53 pp | 0.9008 ± 0.0317 |
| 15 | 8.29% ± 1.87 pp | 47.00% ± 16.36 pp | 0.9013 ± 0.0296 |

- **R2L:** il vantaggio delle profondità 13/15 si conferma nei dieci split:
  gli errori medi scendono dal 19,45% all'8,74% e all'8,29%.
- **U2R:** gli errori medi sono 55%, 49% e 47%. In tutte e tre le configurazioni
  le singole prove vanno dal 30% al 70% di errori. Il precedente seed 42, con 30%,
  era uno degli esiti più favorevoli osservati, non la prestazione tipica.
- **Precision U2R media:** 77,08% a 10, 69,11% a 13 e 64,70% a 15. Con gli alberi
  più profondi il recall medio sale (45% / 51% / 53%), ma aumentano i falsi positivi.
- **13 contro 15:** il macro F1 medio differisce di circa 0,0005; 15 supera 13
  in 4 split su 10, mentre 13 lo supera negli altri 6. È un confronto descrittivo:
  non emerge un vincitore stabile dalle medie e da queste dieci prove sovrapposte.

Ogni validation contiene 10 U2R; in totale abbiamo valutato **43 record U2R distinti**,
riutilizzati fra gli split, non 100 casi indipendenti. Le dieci prove rendono visibile
la variabilità; non risolvono la scarsità di dati o il rischio di generalizzazione.
Non scegliamo automaticamente un modello finale e non avviamo altri esperimenti.


## Interpretazione

Le ripetizioni riducono la dipendenza da un unico split fortunato o sfavorevole.
Media e dispersione descrivono la stabilità osservata **su questo dataset e protocollo**;
non garantiscono prestazioni affidabili su reti nuove. Con pochi esempi U2R, gli
estremi osservati e il numero di casi distinti sono importanti quanto la media.

Nessun dataset aggiunto, feature rimossa, bilanciamento o modello diverso. Il test
ufficiale resta separato. Prima di altre modifiche confrontiamo queste medie e la
variabilità delle tre profondità. Riepilogo in
[REPEATED_VALIDATION_RESULTS.md](../docs/REPEATED_VALIDATION_RESULTS.md).
